In [5]:
from pathlib import Path
import pandas as pd

CF_DATA = Path("cf_data")   # change if needed
engle_path = CF_DATA / "engle_clean_star_rows_only.csv"

df = pd.read_csv(engle_path)
print(df.shape)
df.head()

(380, 7)


,source_paper,star_name,normalized_name,age_gyr,age_err_lo_gyr,age_err_hi_gyr,prot_days
0,Engle_2023,L266-195,L266-195,2.50,0.93,1.82,23.19
1,Engle_2023,LP 856-54,LP 856-54,2.52,0.61,1.58,23.27
2,Engle_2023,G59-39,G59-39,3.20,0.93,1.96,33.82
3,Engle_2023,LP 775-52,LP 775-52,3.39,0.20,0.29,36.90
4,Engle_2023,GJ 2131 A,GJ 2131 A,3.47,1.69,1.69,34.82


In [1]:
import re
import pandas as pd

def prep_name(name: str) -> str:
    if pd.isna(name):
        return ""
    s = str(name).strip()

    # normalize whitespace
    s = re.sub(r"\s+", " ", s)

    # optional: keep A/B labels, since they matter
    # just return as-is for now
    return s

df["query_name"] = df["star_name"].apply(prep_name)

NameError: name 'df' is not defined

In [7]:
%pip install astroquery astropy

Note: you may need to restart the kernel to use updated packages.


In [8]:
from astroquery.simbad import Simbad
import pandas as pd

custom_simbad = Simbad()
custom_simbad.add_votable_fields("ra(d)", "dec(d)", "ids")

def simbad_lookup(name):
    try:
        res = custom_simbad.query_object(name)
        if res is None or len(res) == 0:
            return {
                "simbad_found": False,
                "simbad_main_id": None,
                "simbad_ra_deg": None,
                "simbad_dec_deg": None,
                "simbad_ids": None,
            }

        row = res[0]
        ids = row["IDS"] if "IDS" in row.colnames else None

        return {
            "simbad_found": True,
            "simbad_main_id": str(row["MAIN_ID"]).strip(),
            "simbad_ra_deg": float(row["RA_d"]),
            "simbad_dec_deg": float(row["DEC_d"]),
            "simbad_ids": str(ids) if ids is not None else None,
        }
    except Exception as e:
        return {
            "simbad_found": False,
            "simbad_main_id": None,
            "simbad_ra_deg": None,
            "simbad_dec_deg": None,
            "simbad_ids": None,
            "simbad_error": str(e),
        }

C:\Users\caram\AppData\Local\Temp\ipykernel_42332\4203776614.py:5: DeprecationWarning: 'ra(d)' has been renamed 'ra'. You'll see it appearing with its new name in the output table
  custom_simbad.add_votable_fields("ra(d)", "dec(d)", "ids")
C:\Users\caram\AppData\Local\Temp\ipykernel_42332\4203776614.py:5: DeprecationWarning: 'dec(d)' has been renamed 'dec'. You'll see it appearing with its new name in the output table
  custom_simbad.add_votable_fields("ra(d)", "dec(d)", "ids")


In [9]:
test = df.head(20).copy()
simbad_rows = test["query_name"].apply(simbad_lookup).apply(pd.Series)
test = pd.concat([test, simbad_rows], axis=1)
test[["star_name", "simbad_found", "simbad_main_id", "simbad_ra_deg", "simbad_dec_deg"]]

,star_name,simbad_found,simbad_main_id,simbad_ra_deg,simbad_dec_deg
0,L266-195,False,None,None,None
1,LP 856-54,False,None,None,None
2,G59-39,False,None,None,None
3,LP 775-52,False,None,None,None
4,GJ 2131 A,False,None,None,None
5,G111-72,False,None,None,None
6,HIP 43232 B,False,None,None,None
7,LP 587-54,False,None,None,None
8,GJ 366,False,None,None,None
9,GJ 328,False,None,None,None


In [10]:
import re

def extract_gaia_dr3_id(ids_str):
    if not isinstance(ids_str, str):
        return None

    m = re.search(r"Gaia DR3\s+(\d+)", ids_str)
    if m:
        return m.group(1)

    return None

In [12]:
test["gaia_dr3_source_id_from_simbad"] = test["simbad_ids"].apply(extract_gaia_dr3_id)

test

,source_paper,star_name,normalized_name,age_gyr,age_err_lo_gyr,age_err_hi_gyr,prot_days,query_name,simbad_found,simbad_main_id,simbad_ra_deg,simbad_dec_deg,simbad_ids,simbad_error,gaia_dr3_source_id_from_simbad
0,Engle_2023,L266-195,L266-195,2.50,0.93,1.82,23.19,L266-195,False,None,None,None,None,'MAIN_ID',None
1,Engle_2023,LP 856-54,LP 856-54,2.52,0.61,1.58,23.27,LP 856-54,False,None,None,None,None,'MAIN_ID',None
2,Engle_2023,G59-39,G59-39,3.20,0.93,1.96,33.82,G59-39,False,None,None,None,None,'MAIN_ID',None
3,Engle_2023,LP 775-52,LP 775-52,3.39,0.20,0.29,36.90,LP 775-52,False,None,None,None,None,'MAIN_ID',None
4,Engle_2023,GJ 2131 A,GJ 2131 A,3.47,1.69,1.69,34.82,GJ 2131 A,False,None,None,None,None,'MAIN_ID',None
5,Engle_2023,G111-72,G111-72,3.64,0.68,1.35,38.96,G111-72,False,None,None,None,None,'MAIN_ID',None
6,Engle_2023,HIP 43232 B,HIP 43232 B,3.95,0.35,0.35,41.30,HIP 43232 B,False,None,None,None,None,'MAIN_ID',None
7,Engle_2023,LP 587-54,LP 587-54,6.34,1.38,1.49,59.50,LP 587-54,False,None,None,None,None,'MAIN_ID',None
8,Engle_2023,GJ 366,GJ 366,9.50,1.50,1.50,87.10,GJ 366,False,None,None,None,None,'MAIN_ID',None
9,Engle_2023,GJ 328,GJ 328,9.50,1.50,1.50,80.50,GJ 328,False,None,None,None,None,'MAIN_ID',None


In [13]:
from astroquery.gaia import Gaia

def gaia_from_source_id(source_id):
    if source_id is None or pd.isna(source_id):
        return None

    query = f"""
    SELECT
        gs.source_id,
        gs.ra,
        gs.dec,
        gs.parallax,
        gs.parallax_error,
        gs.pmra,
        gs.pmdec,
        gs.phot_g_mean_mag,
        gs.phot_bp_mean_mag,
        gs.phot_rp_mean_mag,
        gs.bp_rp,
        gs.g_rp,
        gs.ruwe,
        ap.ebpminrp_gspphot
    FROM gaiadr3.gaia_source AS gs
    LEFT JOIN gaiadr3.astrophysical_parameters AS ap
      ON gs.source_id = ap.source_id
    WHERE gs.source_id = {source_id}
    """
    job = Gaia.launch_job(query)
    r = job.get_results()
    if len(r) == 0:
        return None
    return r.to_pandas().iloc[0].to_dict()

The archive is unstable and may perform below expectations. If launching multiple, consecutive, heavy queries through Python, please space them out (e.g., using sleep(1)) to avoid overloading the system. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the December 2025 infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive


In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

def gaia_cone_search(ra_deg, dec_deg, radius_arcsec=5):
    if pd.isna(ra_deg) or pd.isna(dec_deg):
        return None

    coord = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
    radius = u.Quantity(radius_arcsec, u.arcsec)

    query = f"""
    SELECT TOP 10
        gs.source_id,
        gs.ra,
        gs.dec,
        gs.parallax,
        gs.parallax_error,
        gs.pmra,
        gs.pmdec,
        gs.phot_g_mean_mag,
        gs.phot_bp_mean_mag,
        gs.phot_rp_mean_mag,
        gs.bp_rp,
        gs.g_rp,
        gs.ruwe,
        ap.ebpminrp_gspphot,
        DISTANCE(
            POINT('ICRS', gs.ra, gs.dec),
            POINT('ICRS', {ra_deg}, {dec_deg})
        ) * 3600.0 AS dist_arcsec
    FROM gaiadr3.gaia_source AS gs
    LEFT JOIN gaiadr3.astrophysical_parameters AS ap
      ON gs.source_id = ap.source_id
    WHERE 1 = CONTAINS(
        POINT('ICRS', gs.ra, gs.dec),
        CIRCLE('ICRS', {ra_deg}, {dec_deg}, {radius.to(u.deg).value})
    )
    ORDER BY dist_arcsec ASC
    """
    job = Gaia.launch_job(query)
    r = job.get_results()
    if len(r) == 0:
        return None
    return r.to_pandas()

In [ ]:
def choose_best_gaia_match(candidates):
    if candidates is None or len(candidates) == 0:
        return None, "no_gaia_match"

    best = candidates.sort_values("dist_arcsec").iloc[0].to_dict()

    n_close = (candidates["dist_arcsec"] <= 2.0).sum()
    flags = []

    if n_close > 1:
        flags.append("multiple_close_matches")
    if pd.notna(best.get("ruwe")) and best["ruwe"] >= 1.2:
        flags.append("high_ruwe")
    if pd.isna(best.get("ebpminrp_gspphot")):
        flags.append("missing_reddening")

    flag = ";".join(flags) if flags else "ok"
    return best, flag

In [16]:
def resolve_engle_star(row):
    out = {
        "simbad_main_id": None,
        "simbad_ra_deg": None,
        "simbad_dec_deg": None,
        "gaia_dr3_source_id": None,
        "match_sep_arcsec": None,
        "ra": None,
        "dec": None,
        "parallax": None,
        "pmra": None,
        "pmdec": None,
        "phot_g_mean_mag": None,
        "phot_bp_mean_mag": None,
        "phot_rp_mean_mag": None,
        "bp_rp": None,
        "g_rp": None,
        "ruwe": None,
        "ebpminrp_gspphot": None,
        "bp_rp_0": None,
        "match_flag": None,
    }

    sim = simbad_lookup(row["query_name"])
    out["simbad_main_id"] = sim.get("simbad_main_id")
    out["simbad_ra_deg"] = sim.get("simbad_ra_deg")
    out["simbad_dec_deg"] = sim.get("simbad_dec_deg")

    gaia_id = extract_gaia_dr3_id(sim.get("simbad_ids"))
    if gaia_id is not None:
        g = gaia_from_source_id(gaia_id)
        if g is not None:
            out["gaia_dr3_source_id"] = g.get("source_id")
            out["ra"] = g.get("ra")
            out["dec"] = g.get("dec")
            out["parallax"] = g.get("parallax")
            out["pmra"] = g.get("pmra")
            out["pmdec"] = g.get("pmdec")
            out["phot_g_mean_mag"] = g.get("phot_g_mean_mag")
            out["phot_bp_mean_mag"] = g.get("phot_bp_mean_mag")
            out["phot_rp_mean_mag"] = g.get("phot_rp_mean_mag")
            out["bp_rp"] = g.get("bp_rp")
            out["g_rp"] = g.get("g_rp")
            out["ruwe"] = g.get("ruwe")
            out["ebpminrp_gspphot"] = g.get("ebpminrp_gspphot")
            if pd.notna(out["bp_rp"]) and pd.notna(out["ebpminrp_gspphot"]):
                out["bp_rp_0"] = out["bp_rp"] - out["ebpminrp_gspphot"]
            out["match_flag"] = "matched_via_simbad_gaia_id"
            return pd.Series(out)

    candidates = gaia_cone_search(sim.get("simbad_ra_deg"), sim.get("simbad_dec_deg"), radius_arcsec=5)
    best, flag = choose_best_gaia_match(candidates)

    if best is None:
        out["match_flag"] = flag
        return pd.Series(out)

    out["gaia_dr3_source_id"] = best.get("source_id")
    out["match_sep_arcsec"] = best.get("dist_arcsec")
    out["ra"] = best.get("ra")
    out["dec"] = best.get("dec")
    out["parallax"] = best.get("parallax")
    out["pmra"] = best.get("pmra")
    out["pmdec"] = best.get("pmdec")
    out["phot_g_mean_mag"] = best.get("phot_g_mean_mag")
    out["phot_bp_mean_mag"] = best.get("phot_bp_mean_mag")
    out["phot_rp_mean_mag"] = best.get("phot_rp_mean_mag")
    out["bp_rp"] = best.get("bp_rp")
    out["g_rp"] = best.get("g_rp")
    out["ruwe"] = best.get("ruwe")
    out["ebpminrp_gspphot"] = best.get("ebpminrp_gspphot")

    if pd.notna(out["bp_rp"]) and pd.notna(out["ebpminrp_gspphot"]):
        out["bp_rp_0"] = out["bp_rp"] - out["ebpminrp_gspphot"]

    out["match_flag"] = flag
    return pd.Series(out)

In [17]:
sample = df.head(25).copy()
resolved = sample.apply(resolve_engle_star, axis=1)
sample_out = pd.concat([sample, resolved], axis=1)

sample_out[[
    "star_name",
    "simbad_main_id",
    "gaia_dr3_source_id",
    "match_sep_arcsec",
    "bp_rp",
    "g_rp",
    "ebpminrp_gspphot",
    "bp_rp_0",
    "ruwe",
    "match_flag",
]]

,star_name,simbad_main_id,gaia_dr3_source_id,match_sep_arcsec,bp_rp,g_rp,ebpminrp_gspphot,bp_rp_0,ruwe,match_flag
0,L266-195,None,None,None,None,None,None,None,None,no_gaia_match
1,LP 856-54,None,None,None,None,None,None,None,None,no_gaia_match
2,G59-39,None,None,None,None,None,None,None,None,no_gaia_match
3,LP 775-52,None,None,None,None,None,None,None,None,no_gaia_match
4,GJ 2131 A,None,None,None,None,None,None,None,None,no_gaia_match
5,G111-72,None,None,None,None,None,None,None,None,no_gaia_match
6,HIP 43232 B,None,None,None,None,None,None,None,None,no_gaia_match
7,LP 587-54,None,None,None,None,None,None,None,None,no_gaia_match
8,GJ 366,None,None,None,None,None,None,None,None,no_gaia_match
9,GJ 328,None,None,None,None,None,None,None,None,no_gaia_match


In [18]:
from astroquery.simbad import Simbad

s = Simbad()
s.add_votable_fields("ra(d)", "dec(d)", "ids")

res = s.query_object("GJ 366")
print(res)

C:\Users\caram\AppData\Local\Temp\ipykernel_42332\1904562137.py:4: DeprecationWarning: 'ra(d)' has been renamed 'ra'. You'll see it appearing with its new name in the output table
  s.add_votable_fields("ra(d)", "dec(d)", "ids")
C:\Users\caram\AppData\Local\Temp\ipykernel_42332\1904562137.py:4: DeprecationWarning: 'dec(d)' has been renamed 'dec'. You'll see it appearing with its new name in the output table
  s.add_votable_fields("ra(d)", "dec(d)", "ids")


  main_id           ra        ... matched_id
                   deg        ...           
----------- ----------------- ... ----------
BD+76  3952 146.7020252337433 ...     GJ 366


In [20]:
from astroquery.simbad import Simbad

s = Simbad()
s.add_votable_fields("ra(d)", "dec(d)", "ids")

res = s.query_object("GJ 366")
print(res.colnames)
print(res[0])

C:\Users\caram\AppData\Local\Temp\ipykernel_42332\359297842.py:4: DeprecationWarning: 'ra(d)' has been renamed 'ra'. You'll see it appearing with its new name in the output table
  s.add_votable_fields("ra(d)", "dec(d)", "ids")
C:\Users\caram\AppData\Local\Temp\ipykernel_42332\359297842.py:4: DeprecationWarning: 'dec(d)' has been renamed 'dec'. You'll see it appearing with its new name in the output table
  s.add_votable_fields("ra(d)", "dec(d)", "ids")


['main_id', 'ra', 'dec', 'coo_err_maj', 'coo_err_min', 'coo_err_angle', 'coo_wavelength', 'coo_bibcode', 'ids', 'matched_id']
  main_id           ra               dec        coo_err_maj coo_err_min coo_err_angle coo_wavelength     coo_bibcode                                                                                                                                                                                                                                                                                  ids                                                                                                                                                                                                                                                                              matched_id
                   deg               deg            mas         mas          deg                                                                                                                        

In [ ]:
from astroquery.simbad import Simbad
import pandas as pd

custom_simbad = Simbad()
custom_simbad.add_votable_fields("ra(d)", "dec(d)", "ids")

def _get_first_existing(row, possible_names):
    for name in possible_names:
        if name in row.colnames:
            val = row[name]
            if val is not None:
                return val
    return None

def simbad_lookup(name):
    if not isinstance(name, str) or not name.strip():
        return {
            "simbad_found": False,
            "simbad_main_id": None,
            "simbad_ra_deg": None,
            "simbad_dec_deg": None,
            "simbad_ids": None,
            "simbad_status": "empty_name",
        }

    try:
        res = custom_simbad.query_object(name)

        if res is None or len(res) == 0:
            return {
                "simbad_found": False,
                "simbad_main_id": None,
                "simbad_ra_deg": None,
                "simbad_dec_deg": None,
                "simbad_ids": None,
                "simbad_status": "no_result",
            }

        row = res[0]

        main_id = _get_first_existing(row, ["MAIN_ID", "main_id"])
        ra_val   = _get_first_existing(row, ["RA_d", "ra", "RA"])
        dec_val  = _get_first_existing(row, ["DEC_d", "dec", "DEC"])
        ids_val  = _get_first_existing(row, ["IDS", "ids"])

        return {
            "simbad_found": True,
            "simbad_main_id": str(main_id).strip() if main_id is not None else None,
            "simbad_ra_deg": float(ra_val) if ra_val is not None else None,
            "simbad_dec_deg": float(dec_val) if dec_val is not None else None,
            "simbad_ids": str(ids_val) if ids_val is not None else None,
            "simbad_status": "ok",
        }

    except Exception as e:
        return {
            "simbad_found": False,
            "simbad_main_id": None,
            "simbad_ra_deg": None,
            "simbad_dec_deg": None,
            "simbad_ids": None,
            "simbad_status": f"error: {e}",
        }

In [22]:
test_names = ["GJ 366", "GJ 821", "HIP 43232 B", "LP 856-54"]

for name in test_names:
    print(name, simbad_lookup(name))

GJ 366 {'simbad_found': True, 'simbad_main_id': 'BD+76  3952', 'simbad_ra_deg': 146.7020252337433, 'simbad_dec_deg': 76.04391682018667, 'simbad_ids': 'TIC 219431676|Karmn J09468+760|GJ 366|HIP 47981|Gaia DR3 1130450270073924608|CNS5 2420|PLX 2302|LSPM J0946+7602|ASCC   30041|AC +76  3952|AC +76  3952-0|CSI+76-09423|CSI+76-09417|Ci 20  545|G 252-44|G 253-30|GCRV 60845|GEN# +9.80252044|HIC  47981|LFT  679|LHS  2188|LTT 12585|MCC 120|NLTT 22468|PM 09417+7617|Ross  434|TYC 4542-2612-1|VVO 134|[RHG95]  1537|2MASS J09464844+7602388|USNO-B1.0 1660-00055540|PLX 2302.00|BD+76  3952|GCRV  3533 E|UCAC4 831-011894|PM J09468+7602|Gaia DR1 1130450270073924608|WEB  8978|Gaia DR2 1130450270073924608', 'simbad_status': 'ok'}
GJ 821 {'simbad_found': True, 'simbad_main_id': 'Wolf  918', 'simbad_ra_deg': 317.32261313117283, 'simbad_dec_deg': -13.3025036466225, 'simbad_ids': 'TIC 214576902|StKM 2-1498|GJ 821|Gaia DR3 6885776098199761024|PLX 5084|CNS5 5223|CSI-13-21065|Ci 20 1261|GEN# +6.10140918|GSC 05783-

In [23]:
tmp = df.head(20).copy()
tmp["query_name"] = tmp["star_name"]

sim = tmp["query_name"].apply(simbad_lookup).apply(pd.Series)
tmp = pd.concat([tmp, sim], axis=1)

tmp[["star_name", "simbad_found", "simbad_main_id", "simbad_ra_deg", "simbad_dec_deg", "simbad_ids", "simbad_status"]]

,star_name,simbad_found,simbad_main_id,simbad_ra_deg,simbad_dec_deg,simbad_ids,simbad_status
0,L266-195,True,L 266-195,246.759148,-54.197447,TIC 220978213|Gaia DR3 5932005595773303040|TYC...,ok
1,LP 856-54,True,L 619-49,207.846787,-27.564567,TIC 72511761|Gaia DR3 6177238671977087360|CCDM...,ok
2,G59-39,True,G 59-39,191.805960,14.703900,TIC 92043862|Gaia DR3 3930922058356114432|ASCC...,ok
3,LP 775-52,True,LP 775-52,65.431051,-16.000157,TIC 71011069|Gaia DR3 3175193288128400768|2MAS...,ok
4,GJ 2131 A,True,G 154-B5A,266.472296,-13.306205,TIC 419437827|WISE J174553.39-131821.3|Gaia DR...,ok
5,G111-72,True,G 111-72,125.017625,38.578088,TIC 18783116|Gaia DR3 908364387441657472|AP J0...,ok
6,HIP 43232 B,True,UCAC3 222-97968,132.103835,20.705235,Gaia DR3 661794060190112000|UCAC3 222-97968|UC...,ok
7,LP 587-54,True,G 271-52,20.815330,-2.145895,TIC 248949078|Gaia DR3 2485305370114328576|2MA...,ok
8,GJ 366,True,BD+76 3952,146.702025,76.043917,TIC 219431676|Karmn J09468+760|GJ 366|HIP 4798...,ok
9,GJ 328,True,BD+02 2098,133.781757,1.546504,TIC 265373654|StKM 2-525|HIP 43790|Gaia DR3 57...,ok


In [26]:
import re
import pandas as pd

def extract_gaia_dr3_id(ids_str):
    if not isinstance(ids_str, str) or not ids_str.strip():
        return None

    m = re.search(r"Gaia DR3\s+(\d+)", ids_str)
    if m:
        return m.group(1)

    return None

In [28]:
print(df.columns.tolist())

['source_paper', 'star_name', 'normalized_name', 'age_gyr', 'age_err_lo_gyr', 'age_err_hi_gyr', 'prot_days', 'query_name']


In [29]:
sim = df["query_name"].apply(simbad_lookup).apply(pd.Series)
print(sim.columns.tolist())
sim.head()

['simbad_found', 'simbad_main_id', 'simbad_ra_deg', 'simbad_dec_deg', 'simbad_ids', 'simbad_status']


,simbad_found,simbad_main_id,simbad_ra_deg,simbad_dec_deg,simbad_ids,simbad_status
0,True,L 266-195,246.759148,-54.197447,TIC 220978213|Gaia DR3 5932005595773303040|TYC...,ok
1,True,L 619-49,207.846787,-27.564567,TIC 72511761|Gaia DR3 6177238671977087360|CCDM...,ok
2,True,G 59-39,191.805960,14.703900,TIC 92043862|Gaia DR3 3930922058356114432|ASCC...,ok
3,True,LP 775-52,65.431051,-16.000157,TIC 71011069|Gaia DR3 3175193288128400768|2MAS...,ok
4,True,G 154-B5A,266.472296,-13.306205,TIC 419437827|WISE J174553.39-131821.3|Gaia DR...,ok


In [30]:
df = pd.concat([df, sim], axis=1)
print(df.columns.tolist())

['source_paper', 'star_name', 'normalized_name', 'age_gyr', 'age_err_lo_gyr', 'age_err_hi_gyr', 'prot_days', 'query_name', 'simbad_found', 'simbad_main_id', 'simbad_ra_deg', 'simbad_dec_deg', 'simbad_ids', 'simbad_status']


In [31]:
df["gaia_dr3_source_id_from_simbad"] = df["simbad_ids"].apply(extract_gaia_dr3_id)

In [32]:
df[[
    "star_name",
    "simbad_main_id",
    "simbad_ids",
    "gaia_dr3_source_id_from_simbad"
]].head(20)

,star_name,simbad_main_id,simbad_ids,gaia_dr3_source_id_from_simbad
0,L266-195,L 266-195,TIC 220978213|Gaia DR3 5932005595773303040|TYC...,5932005595773303040
1,LP 856-54,L 619-49,TIC 72511761|Gaia DR3 6177238671977087360|CCDM...,6177238671977087360
2,G59-39,G 59-39,TIC 92043862|Gaia DR3 3930922058356114432|ASCC...,3930922058356114432
3,LP 775-52,LP 775-52,TIC 71011069|Gaia DR3 3175193288128400768|2MAS...,3175193288128400768
4,GJ 2131 A,G 154-B5A,TIC 419437827|WISE J174553.39-131821.3|Gaia DR...,4149820284980989184
5,G111-72,G 111-72,TIC 18783116|Gaia DR3 908364387441657472|AP J0...,908364387441657472
6,HIP 43232 B,UCAC3 222-97968,Gaia DR3 661794060190112000|UCAC3 222-97968|UC...,661794060190112000
7,LP 587-54,G 271-52,TIC 248949078|Gaia DR3 2485305370114328576|2MA...,2485305370114328576
8,GJ 366,BD+76 3952,TIC 219431676|Karmn J09468+760|GJ 366|HIP 4798...,1130450270073924608
9,GJ 328,BD+02 2098,TIC 265373654|StKM 2-525|HIP 43790|Gaia DR3 57...,577602496345490176


In [34]:
import pandas as pd

df = pd.read_csv("cf_data/engle_clean_star_rows_only.csv")
df["query_name"] = df["star_name"].apply(prep_name)
sim = df["query_name"].apply(simbad_lookup).apply(pd.Series)
df = pd.concat([df, sim], axis=1)
df["gaia_dr3_source_id_from_simbad"] = df["simbad_ids"].apply(extract_gaia_dr3_id)

In [35]:
df

,source_paper,star_name,normalized_name,age_gyr,age_err_lo_gyr,age_err_hi_gyr,prot_days,query_name,simbad_found,simbad_main_id,simbad_ra_deg,simbad_dec_deg,simbad_ids,simbad_status,gaia_dr3_source_id_from_simbad
0,Engle_2023,L266-195,L266-195,2.50,0.93,1.82,23.19,L266-195,True,L 266-195,246.759148,-54.197447,TIC 220978213|Gaia DR3 5932005595773303040|TYC...,ok,5932005595773303040
1,Engle_2023,LP 856-54,LP 856-54,2.52,0.61,1.58,23.27,LP 856-54,True,L 619-49,207.846787,-27.564567,TIC 72511761|Gaia DR3 6177238671977087360|CCDM...,ok,6177238671977087360
2,Engle_2023,G59-39,G59-39,3.20,0.93,1.96,33.82,G59-39,True,G 59-39,191.805960,14.703900,TIC 92043862|Gaia DR3 3930922058356114432|ASCC...,ok,3930922058356114432
3,Engle_2023,LP 775-52,LP 775-52,3.39,0.20,0.29,36.90,LP 775-52,True,LP 775-52,65.431051,-16.000157,TIC 71011069|Gaia DR3 3175193288128400768|2MAS...,ok,3175193288128400768
4,Engle_2023,GJ 2131 A,GJ 2131 A,3.47,1.69,1.69,34.82,GJ 2131 A,True,G 154-B5A,266.472296,-13.306205,TIC 419437827|WISE J174553.39-131821.3|Gaia DR...,ok,4149820284980989184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,Engle_2024,GJ 581,GJ 581,NaN,NaN,NaN,147.80,GJ 581,True,BD-07 4003,229.861779,-7.722275,TIC 36853511|Karmn J15194-077|MCC 159|StKM 2-1...,ok,6322070093095493504
376,Engle_2024,GJ 699,GJ 699,NaN,NaN,NaN,149.50,GJ 699,True,NAME Barnard's star,269.452077,4.693365,TIC 325554331|MCC 799|StKM 2-1355|GJ 699|HIP 8...,ok,4472832130942575872
377,Engle_2024,GJ 273,GJ 273,NaN,NaN,NaN,160.83,GJ 273,True,BD+05 1668,111.852079,5.225789,AP J07272450+0513329|MCC 17|GJ 273|HIP 36208|G...,ok,3139847906307949696
378,Engle_2024,LP 855-60,LP 855-60,NaN,NaN,NaN,154.89,LP 855-60,True,LP 855-60,206.087105,-26.309772,TIC 101824432|Gaia DR3 6190352001701947520|CNS...,ok,6190352001701947520


In [ ]:
import re

def extract_gaia_dr3_id(ids_str):
    if not isinstance(ids_str, str) or not ids_str.strip():
        return None

    m = re.search(r"Gaia DR3\s+(\d+)", ids_str)
    if m:
        return m.group(1)

    return None

In [37]:
df["gaia_dr3_source_id_from_simbad"] = df["simbad_ids"].apply(extract_gaia_dr3_id)

In [38]:
df[[
    "star_name",
    "simbad_main_id",
    "gaia_dr3_source_id_from_simbad"
]].head(20)

,star_name,simbad_main_id,gaia_dr3_source_id_from_simbad
0,L266-195,L 266-195,5932005595773303040
1,LP 856-54,L 619-49,6177238671977087360
2,G59-39,G 59-39,3930922058356114432
3,LP 775-52,LP 775-52,3175193288128400768
4,GJ 2131 A,G 154-B5A,4149820284980989184
5,G111-72,G 111-72,908364387441657472
6,HIP 43232 B,UCAC3 222-97968,661794060190112000
7,LP 587-54,G 271-52,2485305370114328576
8,GJ 366,BD+76 3952,1130450270073924608
9,GJ 328,BD+02 2098,577602496345490176


In [39]:
df["gaia_dr3_source_id_from_simbad"].notna().sum(), len(df)

(np.int64(329), 380)

In [40]:
matched_direct = df[df["gaia_dr3_source_id_from_simbad"].notna()].copy()
needs_cone = df[df["gaia_dr3_source_id_from_simbad"].isna()].copy()

In [41]:
direct_ids = (
    matched_direct["gaia_dr3_source_id_from_simbad"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

len(direct_ids)

261

In [42]:
from astroquery.gaia import Gaia
import pandas as pd

def chunk_list(lst, n=200):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def query_gaia_by_source_ids(source_ids):
    ids_str = ",".join(source_ids)

    query = f"""
    SELECT
        gs.source_id,
        gs.ra,
        gs.dec,
        gs.parallax,
        gs.parallax_error,
        gs.pmra,
        gs.pmdec,
        gs.phot_g_mean_mag,
        gs.phot_bp_mean_mag,
        gs.phot_rp_mean_mag,
        gs.bp_rp,
        gs.g_rp,
        gs.ruwe,
        ap.ebpminrp_gspphot
    FROM gaiadr3.gaia_source AS gs
    LEFT JOIN gaiadr3.astrophysical_parameters AS ap
      ON gs.source_id = ap.source_id
    WHERE gs.source_id IN ({ids_str})
    """

    job = Gaia.launch_job(query)
    r = job.get_results()
    return r.to_pandas()

In [43]:
gaia_parts = []

for chunk in chunk_list(direct_ids, n=200):
    part = query_gaia_by_source_ids(chunk)
    gaia_parts.append(part)

gaia_direct = pd.concat(gaia_parts, ignore_index=True)
gaia_direct.head()

HTTPError: Error 400: 
Cannot parse query '
    SELECT
 TOP 2000         gs.source_id,
        gs.ra,
        gs.dec,
        gs.parallax,
        gs.parallax_error,
        gs.pmra,
        gs.pmdec,
        gs.phot_g_mean_mag,
        gs.phot_bp_mean_mag,
        gs.phot_rp_mean_mag,
        gs.bp_rp,
        gs.g_rp,
        gs.ruwe,
        ap.ebpminrp_gspphot
    FROM gaiadr3.gaia_source AS gs
    LEFT JOIN gaiadr3.astrophysical_parameters AS ap
      ON gs.source_id = ap.source_id
    WHERE gs.source_id IN (2783324526489670144,170639948323693952,4540061704989418240,3855208897392952192,1803475188714186368,1254695603704323712,2828928008202069376,3089711447388931584,5925209583053212800,5144740907820131840,1229089524081628416,266846631637524864,778947814402602752,1472718211053416320,4364480521350598144,1503625448550796032,3657653114880309248,459661938487952000,3741297293732404352,5075098253634738176,2461728194387221504,6887195533352079616,3530383784971799680,2538451703458011264,31369

In [44]:
from astroquery.gaia import Gaia
import pandas as pd

def query_gaia_by_source_ids(source_ids):
    ids_str = ",".join(source_ids)

    query = f"""
    SELECT
        source_id,
        ra,
        dec,
        parallax,
        parallax_error,
        pmra,
        pmdec,
        phot_g_mean_mag,
        phot_bp_mean_mag,
        phot_rp_mean_mag,
        bp_rp,
        ruwe
    FROM gaiadr3.gaia_source
    WHERE source_id IN ({ids_str})
    """
    job = Gaia.launch_job(query)
    r = job.get_results()
    return r.to_pandas()

In [45]:
def query_astrophysical_by_source_ids(source_ids):
    ids_str = ",".join(source_ids)

    query = f"""
    SELECT
        source_id,
        ebpminrp_gspphot
    FROM gaiadr3.astrophysical_parameters
    WHERE source_id IN ({ids_str})
    """
    job = Gaia.launch_job(query)
    r = job.get_results()
    return r.to_pandas()

In [46]:
gaia_parts = []
ap_parts = []

for chunk in chunk_list(direct_ids, n=200):
    gaia_parts.append(query_gaia_by_source_ids(chunk))
    ap_parts.append(query_astrophysical_by_source_ids(chunk))

gaia_core = pd.concat(gaia_parts, ignore_index=True)
gaia_ap = pd.concat(ap_parts, ignore_index=True)

HTTPError: Error 400: 
Cannot parse query '
    SELECT
 TOP 2000         source_id,
        ebpminrp_gspphot
    FROM gaiadr3.astrophysical_parameters
    WHERE source_id IN (5932005595773303040,6177238671977087360,3930922058356114432,3175193288128400768,4149820284980989184,908364387441657472,661794060190112000,2485305370114328576,1130450270073924608,577602496345490176,6885776098199761024,3817534337625985280,3931313793733241088,222749343414443648,220989197096055040,4201781696994082944,4021566209266336640,5717278911884264576,2871730758921709952,3738752852026813184,3630015546177181952,5402903764121955840,3195919254111314816,1600882199829338368,5853498713190525696,946030529073022080,1996725077535282944,1379500928055726848,975968718869126144,3098328182579892096,3340477717172813568,67198925172337536,3139847906307949696,6322070093095493504,198027271140075136,5115463180914712448,2269520028411874816,3345752830365960320,150197587618319744,3899393459350372224,135885416558354048,19176825266367706

In [47]:
from astroquery.gaia import Gaia

tables = Gaia.load_tables(only_names=True)
table_names = [t.get_qualified_name() for t in tables]

print("num tables:", len(table_names))
print([x for x in table_names if "astrophysical" in x.lower()][:20])
print([x for x in table_names if "gaiadr3" in x.lower()][:50])

INFO: Retrieving tables... [astroquery.utils.tap.core]
500 Error 500:
esavo.tap.TAPException: java.sql.SQLException: esavo.tap.db.DBException: Can not execute the following SQL: 
SELECT argument_name, description, arg_type, default_value, max_value, min_value, arraysize  FROM tap_schema.functions_arguments where all_functions_id=809 ORDER BY arg_order ASC
Because: ERROR: failed to acquire resources on one or more segments
  Detail: could not connect to server: Cannot assign requested address
	Is the server running on host "10.0.3.174" and accepting
	TCP/IP connections on port 55000?
 (seg0 10.0.3.174:55000)


HTTPError: Error 500:
esavo.tap.TAPException: java.sql.SQLException: esavo.tap.db.DBException: Can not execute the following SQL: 
SELECT argument_name, description, arg_type, default_value, max_value, min_value, arraysize  FROM tap_schema.functions_arguments where all_functions_id=809 ORDER BY arg_order ASC
Because: ERROR: failed to acquire resources on one or more segments
  Detail: could not connect to server: Cannot assign requested address
	Is the server running on host "10.0.3.174" and accepting
	TCP/IP connections on port 55000?
 (seg0 10.0.3.174:55000)

In [49]:
try:
    test_query = """
    SELECT TOP 5 *
    FROM gaiadr3.astrophysical_parameters
    """
    job = Gaia.launch_job(test_query)
    res = job.get_results()
    print(res.colnames)
    print(res[:2])
except Exception as e:
    print(type(e).__name__)
    print(e)

['solution_id', 'source_id', 'classprob_dsc_combmod_quasar', 'classprob_dsc_combmod_galaxy', 'classprob_dsc_combmod_star', 'classprob_dsc_combmod_whitedwarf', 'classprob_dsc_combmod_binarystar', 'classprob_dsc_specmod_quasar', 'classprob_dsc_specmod_galaxy', 'classprob_dsc_specmod_star', 'classprob_dsc_specmod_whitedwarf', 'classprob_dsc_specmod_binarystar', 'classprob_dsc_allosmod_quasar', 'classprob_dsc_allosmod_galaxy', 'classprob_dsc_allosmod_star', 'teff_gspphot', 'teff_gspphot_lower', 'teff_gspphot_upper', 'logg_gspphot', 'logg_gspphot_lower', 'logg_gspphot_upper', 'mh_gspphot', 'mh_gspphot_lower', 'mh_gspphot_upper', 'distance_gspphot', 'distance_gspphot_lower', 'distance_gspphot_upper', 'azero_gspphot', 'azero_gspphot_lower', 'azero_gspphot_upper', 'ag_gspphot', 'ag_gspphot_lower', 'ag_gspphot_upper', 'abp_gspphot', 'abp_gspphot_lower', 'abp_gspphot_upper', 'arp_gspphot', 'arp_gspphot_lower', 'arp_gspphot_upper', 'ebpminrp_gspphot', 'ebpminrp_gspphot_lower', 'ebpminrp_gspphot_u

In [51]:
from astroquery.gaia import Gaia
import pandas as pd

def query_gaia_by_source_ids(source_ids):
    ids_str = ",".join(source_ids)

    query = f"""
    SELECT
        source_id,
        ra,
        dec,
        parallax,
        parallax_error,
        pmra,
        pmdec,
        phot_g_mean_mag,
        phot_bp_mean_mag,
        phot_rp_mean_mag,
        bp_rp,
        ruwe
    FROM gaiadr3.gaia_source
    WHERE source_id IN ({ids_str})
    """
    job = Gaia.launch_job(query)
    return job.get_results().to_pandas()

In [52]:
def query_astrophysical_by_source_ids(source_ids):
    ids_str = ",".join(source_ids)

    query = f"""
    SELECT
        source_id,
        ebpminrp_gspphot
    FROM gaiadr3.astrophysical_parameters
    WHERE source_id IN ({ids_str})
    """
    job = Gaia.launch_job(query)
    return job.get_results().to_pandas()

In [55]:
from astroquery.gaia import Gaia

q1 = """
SELECT TOP 5
    source_id,
    phot_g_mean_mag,
    phot_bp_mean_mag,
    phot_rp_mean_mag,
    bp_rp,
    ruwe
FROM gaiadr3.gaia_source
"""
job1 = Gaia.launch_job(q1)
res1 = job1.get_results()
print(res1.colnames)
print(res1[:2])

['source_id', 'phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', 'bp_rp', 'ruwe']
  source_id   phot_g_mean_mag phot_bp_mean_mag ...   bp_rp      ruwe  
                    mag             mag        ...    mag             
------------- --------------- ---------------- ... --------- ---------
1752346816896       19.060192        20.177439 ... 2.0861473 0.9940326
2753074326400       19.975853         21.10666 ... 2.2815895 1.0328494


In [56]:
q2 = """
SELECT TOP 5
    source_id,
    ebpminrp_gspphot
FROM gaiadr3.astrophysical_parameters
"""
job2 = Gaia.launch_job(q2)
res2 = job2.get_results()
print(res2.colnames)
print(res2[:2])

['source_id', 'ebpminrp_gspphot']
  source_id   ebpminrp_gspphot
                    mag       
------------- ----------------
1374389600384           0.1986
3607772690688           0.0153


In [ ]:
from astroquery.gaia import Gaia

def query_gaia_by_source_ids(source_ids):
    ids_str = ",".join(str(x) for x in source_ids)

    query = (
        "SELECT "
        "source_id, ra, dec, parallax, parallax_error, pmra, pmdec, "
        "phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, bp_rp, ruwe "
        "FROM gaiadr3.gaia_source "
        f"WHERE source_id IN ({ids_str})"
    )

    job = Gaia.launch_job(query)
    return job.get_results().to_pandas()


def query_astrophysical_by_source_ids(source_ids):
    ids_str = ",".join(str(x) for x in source_ids)

    query = (
        "SELECT "
        "source_id, ebpminrp_gspphot "
        "FROM gaiadr3.astrophysical_parameters "
        f"WHERE source_id IN ({ids_str})"
    )

    job = Gaia.launch_job(query)
    return job.get_results().to_pandas()

In [ ]:
def query_gaia_one_source_id(source_id):
    query = f"""
    SELECT
        source_id,
        ra,
        dec,
        parallax,
        parallax_error,
        pmra,
        pmdec,
        phot_g_mean_mag,
        phot_bp_mean_mag,
        phot_rp_mean_mag,
        bp_rp,
        ruwe
    FROM gaiadr3.gaia_source
    WHERE source_id = {int(source_id)}
    """
    job = Gaia.launch_job(query)
    r = job.get_results()
    return r.to_pandas()
def query_ap_one_source_id(source_id):
    query = f"""
    SELECT
        source_id,
        ebpminrp_gspphot
    FROM gaiadr3.astrophysical_parameters
    WHERE source_id = {int(source_id)}
    """
    job = Gaia.launch_job(query)
    r = job.get_results()
    return r.to_pandas()

In [66]:
sid = direct_ids[0]

core_one = query_gaia_one_source_id(sid)
ap_one = query_ap_one_source_id(sid)

print(core_one)
print(ap_one)

             source_id          ra        dec   parallax  parallax_error  \
0  5932005595773303040  246.759296 -54.197154  21.820558        0.014829   

        pmra      pmdec  phot_g_mean_mag  phot_bp_mean_mag  phot_rp_mean_mag  \
0  19.540099  65.965484        10.955764         11.769478         10.074646   

      bp_rp      ruwe  
0  1.694832  0.796916  
             source_id  ebpminrp_gspphot
0  5932005595773303040               NaN


In [67]:
import pandas as pd

core_rows = []
ap_rows = []

for i, sid in enumerate(direct_ids, start=1):
    if i % 25 == 0:
        print(f"{i}/{len(direct_ids)}")

    try:
        c = query_gaia_one_source_id(sid)
        if len(c) > 0:
            core_rows.append(c)

        a = query_ap_one_source_id(sid)
        if len(a) > 0:
            ap_rows.append(a)

    except Exception as e:
        print(f"failed for {sid}: {e}")

failed for 6177238671977087360: Error 400: 
Cannot parse query '
    SELECT
 TOP 2000         source_id,
        ra,
        dec,
        parallax,
        parallax_error,
        pmra,
        pmdec,
        phot_g_mean_mag,
        phot_bp_mean_mag,
        phot_rp_mean_mag,
        bp_rp,
        ruwe
    FROM gaiadr3.gaia_source
    WHERE source_id = 6177238671977087360
    ' for job 'aaf3c678-3245-11f1-9054-bc97e148b76b-O': 14 unresolved identifiers: gaia_source [l.15 c.10 - l.15 c.29], source_id [l.3 c.19 - l.3 c.28], ra [l.4 c.9 - l.4 c.11], dec [l.5 c.9 - l.5 c.12], parallax [l.6 c.9 - l.6 c.17], parallax_error [l.7 c.9 - l.7 c.23], pmra [l.8 c.9 - l.8 c.13], pmdec [l.9 c.9 - l.9 c.14], phot_g_mean_mag [l.10 c.9 - l.10 c.24], phot_bp_mean_mag [l.11 c.9 - l.11 c.25], phot_rp_mean_mag [l.12 c.9 - l.12 c.25], bp_rp [l.13 c.9 - l.13 c.14], ruwe [l.14 c.9 - l.14 c.13], source_id [l.16 c.11 - l.16 c.20] !
 - Unknown table "gaiadr3.gaia_source" !
 - Unknown column "source_id" !
 - Unk

[             source_id          ra        dec   parallax  parallax_error  \
 0  5932005595773303040  246.759296 -54.197154  21.820558        0.014829   
 
         pmra      pmdec  phot_g_mean_mag  phot_bp_mean_mag  phot_rp_mean_mag  \
 0  19.540099  65.965484        10.955764         11.769478         10.074646   
 
       bp_rp      ruwe  
 0  1.694832  0.796916  ,
              source_id         ra        dec   parallax  parallax_error  \
 0  3930922058356114432  191.80433  14.704564  16.498496        0.021312   
 
         pmra   pmdec  phot_g_mean_mag  phot_bp_mean_mag  phot_rp_mean_mag  \
 0 -354.72243  149.31         12.77739         13.775156         11.784307   
 
       bp_rp     ruwe  
 0  1.990849  1.22336  ,
              source_id         ra       dec   parallax  parallax_error  \
 0  3175193288128400768  65.432288 -16.00092  21.355441        0.029057   
 
         pmra       pmdec  phot_g_mean_mag  phot_bp_mean_mag  phot_rp_mean_mag  \
 0  267.44587 -171.730706        1

In [69]:
gaia_core = pd.concat(core_rows, ignore_index=True)
gaia_ap = pd.concat(ap_rows, ignore_index=True)

gaia_direct = gaia_core.merge(gaia_ap, how="left", on="source_id")
gaia_direct["g_rp"] = gaia_direct["phot_g_mean_mag"] - gaia_direct["phot_rp_mean_mag"]
gaia_direct["bp_rp_0"] = gaia_direct["bp_rp"] - gaia_direct["ebpminrp_gspphot"]

In [70]:
gaia_direct

,source_id,ra,dec,parallax,parallax_error,pmra,pmdec,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag,bp_rp,ruwe,ebpminrp_gspphot,g_rp,bp_rp_0
0,5932005595773303040,246.759296,-54.197154,21.820558,0.014829,19.540099,65.965484,10.955764,11.769478,10.074646,1.694832,0.796916,NaN,0.881118,NaN
1,3930922058356114432,191.804330,14.704564,16.498496,0.021312,-354.722430,149.310000,12.777390,13.775156,11.784307,1.990849,1.223360,NaN,0.993083,NaN
2,3175193288128400768,65.432288,-16.000920,21.355441,0.029057,267.445870,-171.730706,11.713509,12.747372,10.687969,2.059402,2.017660,NaN,1.025539,NaN
3,908364387441657472,125.016591,38.576947,22.524347,0.018565,-181.910451,-256.608143,12.280472,13.390044,11.235033,2.155011,1.305746,0.0002,1.045439,2.154811
4,2485305370114328576,20.816370,-2.145728,23.151370,0.031866,233.955830,37.652015,11.716210,12.845122,10.654177,2.190946,1.559819,NaN,1.062034,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,6814962601568904576,324.758758,-24.161181,51.399140,0.070609,1003.987424,-710.657765,12.077593,13.643229,10.848627,2.794601,1.540514,NaN,1.228966,NaN
184,5686163724944033792,148.173530,-15.604420,56.820684,0.021463,-118.933799,-134.089874,12.172839,13.689925,10.970720,2.719205,1.240200,NaN,1.202119,NaN
185,6044721476534419584,248.852636,-27.319139,50.826196,0.030425,-5.798222,-894.853226,12.750803,14.414499,11.505672,2.908827,1.249603,0.0001,1.245131,2.908727
186,5329084752471816192,131.159854,-48.085631,68.700905,0.017307,-321.970547,794.451801,12.434480,14.191856,11.165983,3.025873,1.062990,0.0000,1.268497,3.025873


In [71]:
df["gaia_dr3_source_id_from_simbad"] = df["gaia_dr3_source_id_from_simbad"].astype("string")
gaia_direct["source_id"] = gaia_direct["source_id"].astype("string")

df = df.merge(
    gaia_direct,
    how="left",
    left_on="gaia_dr3_source_id_from_simbad",
    right_on="source_id"
)

In [72]:
df[[
    "star_name",
    "gaia_dr3_source_id_from_simbad",
    "bp_rp",
    "g_rp",
    "ruwe",
    "ebpminrp_gspphot",
    "bp_rp_0"
]].head(20)

,star_name,gaia_dr3_source_id_from_simbad,bp_rp,g_rp,ruwe,ebpminrp_gspphot,bp_rp_0
0,L266-195,5932005595773303040,1.694832,0.881118,0.796916,NaN,NaN
1,LP 856-54,6177238671977087360,NaN,NaN,NaN,NaN,NaN
2,G59-39,3930922058356114432,1.990849,0.993083,1.223360,NaN,NaN
3,LP 775-52,3175193288128400768,2.059402,1.025539,2.017660,NaN,NaN
4,GJ 2131 A,4149820284980989184,NaN,NaN,NaN,NaN,NaN
5,G111-72,908364387441657472,2.155011,1.045439,1.305746,0.0002,2.154811
6,HIP 43232 B,661794060190112000,NaN,NaN,NaN,NaN,NaN
7,LP 587-54,2485305370114328576,2.190946,1.062034,1.559819,NaN,NaN
8,GJ 366,1130450270073924608,2.110064,1.035575,1.044251,NaN,NaN
9,GJ 328,577602496345490176,1.832388,0.935581,1.062343,0.0005,1.831888


In [73]:
df["bp_rp"].notna().sum(), len(df)

(np.int64(237), 380)

In [74]:
gaia_direct[["bp_rp", "g_rp", "ruwe", "ebpminrp_gspphot", "bp_rp_0"]].describe()

,bp_rp,g_rp,ruwe,ebpminrp_gspphot,bp_rp_0
count,188.000000,188.000000,185.000000,51.000000,51.000000
mean,2.460768,1.130528,2.547583,0.015529,2.338650
std,0.464516,0.126019,4.562168,0.055836,0.381499
min,1.637033,0.866838,0.796916,0.000000,1.653491
25%,2.091349,1.034778,1.084851,0.000000,2.047809
50%,2.439629,1.138288,1.186639,0.000100,2.293731
75%,2.794490,1.227675,1.449551,0.000250,2.621484
max,4.379336,1.497980,33.123741,0.240500,3.139298


In [75]:
failed_ids = []

core_rows = []
ap_rows = []

for i, sid in enumerate(direct_ids, start=1):
    if i % 25 == 0:
        print(f"{i}/{len(direct_ids)}")

    try:
        c = query_gaia_one_source_id(sid)
        if len(c) > 0:
            core_rows.append(c)

        a = query_ap_one_source_id(sid)
        if len(a) > 0:
            ap_rows.append(a)

    except Exception as e:
        failed_ids.append((sid, str(e)))
        print(f"failed for {sid}: {e}")

failed for 6177238671977087360: Error 400: 
Cannot parse query '
    SELECT
 TOP 2000         source_id,
        ra,
        dec,
        parallax,
        parallax_error,
        pmra,
        pmdec,
        phot_g_mean_mag,
        phot_bp_mean_mag,
        phot_rp_mean_mag,
        bp_rp,
        ruwe
    FROM gaiadr3.gaia_source
    WHERE source_id = 6177238671977087360
    ' for job '91449ff9-3248-11f1-9054-bc97e148b76b-O': 14 unresolved identifiers: gaia_source [l.15 c.10 - l.15 c.29], source_id [l.3 c.19 - l.3 c.28], ra [l.4 c.9 - l.4 c.11], dec [l.5 c.9 - l.5 c.12], parallax [l.6 c.9 - l.6 c.17], parallax_error [l.7 c.9 - l.7 c.23], pmra [l.8 c.9 - l.8 c.13], pmdec [l.9 c.9 - l.9 c.14], phot_g_mean_mag [l.10 c.9 - l.10 c.24], phot_bp_mean_mag [l.11 c.9 - l.11 c.25], phot_rp_mean_mag [l.12 c.9 - l.12 c.25], bp_rp [l.13 c.9 - l.13 c.14], ruwe [l.14 c.9 - l.14 c.13], source_id [l.16 c.11 - l.16 c.20] !
 - Unknown table "gaiadr3.gaia_source" !
 - Unknown column "source_id" !
 - Unk

In [76]:
df

,source_paper,star_name,normalized_name,age_gyr,age_err_lo_gyr,age_err_hi_gyr,prot_days,query_name,simbad_found,simbad_main_id,...,pmra,pmdec,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag,bp_rp,ruwe,ebpminrp_gspphot,g_rp,bp_rp_0
0,Engle_2023,L266-195,L266-195,2.50,0.93,1.82,23.19,L266-195,True,L 266-195,...,19.540099,65.965484,10.955764,11.769478,10.074646,1.694832,0.796916,NaN,0.881118,NaN
1,Engle_2023,LP 856-54,LP 856-54,2.52,0.61,1.58,23.27,LP 856-54,True,L 619-49,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Engle_2023,G59-39,G59-39,3.20,0.93,1.96,33.82,G59-39,True,G 59-39,...,-354.722430,149.310000,12.777390,13.775156,11.784307,1.990849,1.223360,NaN,0.993083,NaN
3,Engle_2023,LP 775-52,LP 775-52,3.39,0.20,0.29,36.90,LP 775-52,True,LP 775-52,...,267.445870,-171.730706,11.713509,12.747372,10.687969,2.059402,2.017660,NaN,1.025539,NaN
4,Engle_2023,GJ 2131 A,GJ 2131 A,3.47,1.69,1.69,34.82,GJ 2131 A,True,G 154-B5A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,Engle_2024,GJ 581,GJ 581,NaN,NaN,NaN,147.80,GJ 581,True,BD-07 4003,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
376,Engle_2024,GJ 699,GJ 699,NaN,NaN,NaN,149.50,GJ 699,True,NAME Barnard's star,...,-801.550978,10362.394207,8.193974,9.791788,6.958091,2.833697,1.084851,0.0,1.235883,2.833697
377,Engle_2024,GJ 273,GJ 273,NaN,NaN,NaN,160.83,GJ 273,True,BD+05 1668,...,571.232347,-3691.486565,8.576388,10.111705,7.353947,2.757758,1.238770,0.0,1.222442,2.757758
378,Engle_2024,LP 855-60,LP 855-60,NaN,NaN,NaN,154.89,LP 855-60,True,LP 855-60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [77]:
failed_ids[:10]

[('6177238671977087360',
  'Error 400: \nCannot parse query \'\n    SELECT\n TOP 2000         source_id,\n        ra,\n        dec,\n        parallax,\n        parallax_error,\n        pmra,\n        pmdec,\n        phot_g_mean_mag,\n        phot_bp_mean_mag,\n        phot_rp_mean_mag,\n        bp_rp,\n        ruwe\n    FROM gaiadr3.gaia_source\n    WHERE source_id = 6177238671977087360\n    \' for job \'91449ff9-3248-11f1-9054-bc97e148b76b-O\': 14 unresolved identifiers: gaia_source [l.15 c.10 - l.15 c.29], source_id [l.3 c.19 - l.3 c.28], ra [l.4 c.9 - l.4 c.11], dec [l.5 c.9 - l.5 c.12], parallax [l.6 c.9 - l.6 c.17], parallax_error [l.7 c.9 - l.7 c.23], pmra [l.8 c.9 - l.8 c.13], pmdec [l.9 c.9 - l.9 c.14], phot_g_mean_mag [l.10 c.9 - l.10 c.24], phot_bp_mean_mag [l.11 c.9 - l.11 c.25], phot_rp_mean_mag [l.12 c.9 - l.12 c.25], bp_rp [l.13 c.9 - l.13 c.14], ruwe [l.14 c.9 - l.14 c.13], source_id [l.16 c.11 - l.16 c.20] !\n - Unknown table "gaiadr3.gaia_source" !\n - Unknown column "

In [78]:
matched_in_gaia_direct = df["gaia_dr3_source_id_from_simbad"].astype("string").isin(
    gaia_direct["source_id"].astype("string")
).sum()

matched_in_gaia_direct

np.int64(237)

In [79]:
expected_ids = set(
    df["gaia_dr3_source_id_from_simbad"]
    .dropna()
    .astype(str)
    .unique()
)

retrieved_ids = set(
    gaia_direct["source_id"]
    .dropna()
    .astype(str)
    .unique()
)

failed_direct_ids = sorted(expected_ids - retrieved_ids)

len(expected_ids), len(retrieved_ids), len(failed_direct_ids)

(261, 188, 73)

In [80]:
failed_direct_df = df[
    df["gaia_dr3_source_id_from_simbad"].astype("string").isin(failed_direct_ids)
].copy()

failed_direct_df[[
    "star_name",
    "simbad_main_id",
    "gaia_dr3_source_id_from_simbad",
    "simbad_ra_deg",
    "simbad_dec_deg"
]].head(30)

,star_name,simbad_main_id,gaia_dr3_source_id_from_simbad,simbad_ra_deg,simbad_dec_deg
1,LP 856-54,L 619-49,6177238671977087360,207.846787,-27.564567
4,GJ 2131 A,G 154-B5A,4149820284980989184,266.472296,-13.306205
6,HIP 43232 B,UCAC3 222-97968,661794060190112000,132.103835,20.705235
10,GJ 821,Wolf 918,6885776098199761024,317.322613,-13.302504
13,LHS 173,Ross 34,222749343414443648,52.221421,37.382447
23,40 Eri C,* omi02 Eri C,3195919254111314816,63.839733,-7.655748
27,2MASS J23095781+5506472,G 233-42,1996725077535282944,347.491082,55.113161
29,LHS 229,G 107-69,975968718869126144,112.678243,48.199608
31,GJ 213,Ross 47,3340477717172813568,85.538615,12.489335
35,GJ 581,BD-07 4003,6322070093095493504,229.861779,-7.722275


In [81]:
failed_direct_df.to_csv("engle_failed_direct_gaia_ids.csv", index=False)

In [82]:
succeeded_direct_ids = sorted(expected_ids & retrieved_ids)

succeeded_direct_df = df[
    df["gaia_dr3_source_id_from_simbad"].astype("string").isin(succeeded_direct_ids)
].copy()

len(succeeded_direct_df)

237

In [83]:
retry_core_rows = []
retry_ap_rows = []
retry_failures = []

for i, sid in enumerate(failed_direct_ids, start=1):
    print(f"{i}/{len(failed_direct_ids)} : {sid}")
    try:
        c = query_gaia_one_source_id(sid)
        if len(c) > 0:
            retry_core_rows.append(c)

        a = query_ap_one_source_id(sid)
        if len(a) > 0:
            retry_ap_rows.append(a)

    except Exception as e:
        retry_failures.append((sid, str(e)))
        print(f"failed again for {sid}: {e}")

1/73 : 1071195187567668480
2/73 : 1149897160435827200
failed again for 1149897160435827200: Error 400: 
Cannot parse query '
    SELECT
 TOP 2000         source_id,
        ra,
        dec,
        parallax,
        parallax_error,
        pmra,
        pmdec,
        phot_g_mean_mag,
        phot_bp_mean_mag,
        phot_rp_mean_mag,
        bp_rp,
        ruwe
    FROM gaiadr3.gaia_source
    WHERE source_id = 1149897160435827200
    ' for job 'ecf6fcb4-3252-11f1-9054-bc97e148b76b-O': 14 unresolved identifiers: gaia_source [l.15 c.10 - l.15 c.29], source_id [l.3 c.19 - l.3 c.28], ra [l.4 c.9 - l.4 c.11], dec [l.5 c.9 - l.5 c.12], parallax [l.6 c.9 - l.6 c.17], parallax_error [l.7 c.9 - l.7 c.23], pmra [l.8 c.9 - l.8 c.13], pmdec [l.9 c.9 - l.9 c.14], phot_g_mean_mag [l.10 c.9 - l.10 c.24], phot_bp_mean_mag [l.11 c.9 - l.11 c.25], phot_rp_mean_mag [l.12 c.9 - l.12 c.25], bp_rp [l.13 c.9 - l.13 c.14], ruwe [l.14 c.9 - l.14 c.13], source_id [l.16 c.11 - l.16 c.20] !
 - Unknown table "g

In [84]:
retry_core = pd.concat(retry_core_rows, ignore_index=True) if retry_core_rows else pd.DataFrame()
retry_ap = pd.concat(retry_ap_rows, ignore_index=True) if retry_ap_rows else pd.DataFrame()

retry_direct = retry_core.merge(retry_ap, how="left", on="source_id")
retry_direct["g_rp"] = retry_direct["phot_g_mean_mag"] - retry_direct["phot_rp_mean_mag"]
retry_direct["bp_rp_0"] = retry_direct["bp_rp"] - retry_direct["ebpminrp_gspphot"]

In [85]:
gaia_direct = pd.concat([gaia_direct, retry_direct], ignore_index=True)
gaia_direct = gaia_direct.drop_duplicates(subset="source_id", keep="first")

In [86]:
retrieved_ids = set(gaia_direct["source_id"].dropna().astype(str).unique())
failed_direct_ids = sorted(expected_ids - retrieved_ids)

len(retrieved_ids), len(failed_direct_ids)

(239, 22)

In [87]:
df["gaia_dr3_source_id_from_simbad"] = df["gaia_dr3_source_id_from_simbad"].astype("string")
gaia_direct["source_id"] = gaia_direct["source_id"].astype("string")

df_direct = df.merge(
    gaia_direct,
    how="left",
    left_on="gaia_dr3_source_id_from_simbad",
    right_on="source_id"
)

df_direct.to_csv("engle_direct_gaia_progress.csv", index=False)

In [88]:
still_failed_direct_df = df[
    df["gaia_dr3_source_id_from_simbad"].astype("string").isin(failed_direct_ids)
].copy()

still_failed_direct_df[[
    "star_name",
    "simbad_main_id",
    "gaia_dr3_source_id_from_simbad",
    "simbad_ra_deg",
    "simbad_dec_deg"
]].head(30)

,star_name,simbad_main_id,gaia_dr3_source_id_from_simbad,simbad_ra_deg,simbad_dec_deg
4,GJ 2131 A,G 154-B5A,4149820284980989184,266.472296,-13.306205
23,40 Eri C,* omi02 Eri C,3195919254111314816,63.839733,-7.655748
27,2MASS J23095781+5506472,G 233-42,1996725077535282944,347.491082,55.113161
29,LHS 229,G 107-69,975968718869126144,112.678243,48.199608
31,GJ 213,Ross 47,3340477717172813568,85.538615,12.489335
109,GJ 2,BD+44 4548,386655019234959872,1.295372,45.786567
111,GJ 47,Wolf 44,426641955043723520,15.333538,61.365745
114,GJ 832,HD 204961,6562924609150908416,323.391563,-49.009000
117,BD-18 359,BD-18 359,5138510181584617856,31.270329,-17.614649
140,HAT 214-03527,2MASS J03303120+2844185,119736889281099136,52.629976,28.738439


In [89]:
still_failed_direct_df[[
    "star_name",
    "simbad_main_id",
    "gaia_dr3_source_id_from_simbad"
]].drop_duplicates().sort_values("star_name")

,star_name,simbad_main_id,gaia_dr3_source_id_from_simbad
172,1RXS J174158.2+475403,ATO J265.4903+47.8980,1363678268537372544
174,2MASS J06170531+8353354,LSPM J0617+8353,1149897160435827200
184,2MASS J13505181+3644168,Ross 1019,1495487688115551744
27,2MASS J23095781+5506472,G 233-42,1996725077535282944
23,40 Eri C,* omi02 Eri C,3195919254111314816
117,BD-18 359,BD-18 359,5138510181584617856
146,G 176-13,G 176-13,778671836983753088
238,GJ 1132,L 320-124,5413438219396893568
109,GJ 2,BD+44 4548,386655019234959872
31,GJ 213,Ross 47,3340477717172813568


In [92]:
retry_ids = (
    still_failed_direct_df["gaia_dr3_source_id_from_simbad"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

len(retry_ids), retry_ids[:10]

(22,
 ['4149820284980989184',
  '3195919254111314816',
  '1996725077535282944',
  '975968718869126144',
  '3340477717172813568',
  '386655019234959872',
  '426641955043723520',
  '6562924609150908416',
  '5138510181584617856',
  '119736889281099136'])

In [95]:
import time
import pandas as pd

retry2_core_rows = []
retry2_ap_rows = []
retry2_failures = []

for i, sid in enumerate(retry_ids, start=1):
    print(f"{i}/{len(retry_ids)} : {sid}")

    try:
        c = query_gaia_one_source_id(sid)
        if len(c) > 0:
            retry2_core_rows.append(c)

        a = query_ap_one_source_id(sid)
        if len(a) > 0:
            retry2_ap_rows.append(a)

        time.sleep(1.0)

    except Exception as e:
        retry2_failures.append((sid, str(e)))
        print(f"failed again for {sid}: {e}")
        time.sleep(2.0)

1/22 : 4149820284980989184
failed again for 4149820284980989184: Error 400: 
Cannot parse query '
    SELECT
 TOP 2000         source_id,
        ra,
        dec,
        parallax,
        parallax_error,
        pmra,
        pmdec,
        phot_g_mean_mag,
        phot_bp_mean_mag,
        phot_rp_mean_mag,
        bp_rp,
        ruwe
    FROM gaiadr3.gaia_source
    WHERE source_id = 4149820284980989184
    ' for job '6e37e250-3254-11f1-9054-bc97e148b76b-O': 14 unresolved identifiers: gaia_source [l.15 c.10 - l.15 c.29], source_id [l.3 c.19 - l.3 c.28], ra [l.4 c.9 - l.4 c.11], dec [l.5 c.9 - l.5 c.12], parallax [l.6 c.9 - l.6 c.17], parallax_error [l.7 c.9 - l.7 c.23], pmra [l.8 c.9 - l.8 c.13], pmdec [l.9 c.9 - l.9 c.14], phot_g_mean_mag [l.10 c.9 - l.10 c.24], phot_bp_mean_mag [l.11 c.9 - l.11 c.25], phot_rp_mean_mag [l.12 c.9 - l.12 c.25], bp_rp [l.13 c.9 - l.13 c.14], ruwe [l.14 c.9 - l.14 c.13], source_id [l.16 c.11 - l.16 c.20] !
 - Unknown table "gaiadr3.gaia_source" !
 - Un

In [96]:
retry2_core = pd.concat(retry2_core_rows, ignore_index=True) if retry2_core_rows else pd.DataFrame()
retry2_ap = pd.concat(retry2_ap_rows, ignore_index=True) if retry2_ap_rows else pd.DataFrame()

if len(retry2_core) > 0:
    retry2_direct = retry2_core.merge(retry2_ap, how="left", on="source_id")
    retry2_direct["g_rp"] = retry2_direct["phot_g_mean_mag"] - retry2_direct["phot_rp_mean_mag"]
    retry2_direct["bp_rp_0"] = retry2_direct["bp_rp"] - retry2_direct["ebpminrp_gspphot"]

    gaia_direct = pd.concat([gaia_direct, retry2_direct], ignore_index=True)
    gaia_direct = gaia_direct.drop_duplicates(subset="source_id", keep="first")

In [97]:
retrieved_ids = set(gaia_direct["source_id"].dropna().astype(str).unique())

still_failed_direct_df = df[
    df["gaia_dr3_source_id_from_simbad"].astype("string").isin(
        sorted(expected_ids - retrieved_ids)
    )
].copy()

len(still_failed_direct_df)

26

In [98]:
df

,source_paper,star_name,normalized_name,age_gyr,age_err_lo_gyr,age_err_hi_gyr,prot_days,query_name,simbad_found,simbad_main_id,...,pmra,pmdec,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag,bp_rp,ruwe,ebpminrp_gspphot,g_rp,bp_rp_0
0,Engle_2023,L266-195,L266-195,2.50,0.93,1.82,23.19,L266-195,True,L 266-195,...,19.540099,65.965484,10.955764,11.769478,10.074646,1.694832,0.796916,NaN,0.881118,NaN
1,Engle_2023,LP 856-54,LP 856-54,2.52,0.61,1.58,23.27,LP 856-54,True,L 619-49,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Engle_2023,G59-39,G59-39,3.20,0.93,1.96,33.82,G59-39,True,G 59-39,...,-354.722430,149.310000,12.777390,13.775156,11.784307,1.990849,1.223360,NaN,0.993083,NaN
3,Engle_2023,LP 775-52,LP 775-52,3.39,0.20,0.29,36.90,LP 775-52,True,LP 775-52,...,267.445870,-171.730706,11.713509,12.747372,10.687969,2.059402,2.017660,NaN,1.025539,NaN
4,Engle_2023,GJ 2131 A,GJ 2131 A,3.47,1.69,1.69,34.82,GJ 2131 A,True,G 154-B5A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,Engle_2024,GJ 581,GJ 581,NaN,NaN,NaN,147.80,GJ 581,True,BD-07 4003,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
376,Engle_2024,GJ 699,GJ 699,NaN,NaN,NaN,149.50,GJ 699,True,NAME Barnard's star,...,-801.550978,10362.394207,8.193974,9.791788,6.958091,2.833697,1.084851,0.0,1.235883,2.833697
377,Engle_2024,GJ 273,GJ 273,NaN,NaN,NaN,160.83,GJ 273,True,BD+05 1668,...,571.232347,-3691.486565,8.576388,10.111705,7.353947,2.757758,1.238770,0.0,1.222442,2.757758
378,Engle_2024,LP 855-60,LP 855-60,NaN,NaN,NaN,154.89,LP 855-60,True,LP 855-60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Cone search fallback + final assembly

Load the intermediate save (stable base after all direct queries), then apply a 5-arcsec cone search for every row still missing a Gaia `source_id` but with known SIMBAD coordinates. Finally add the RUWE quality flag and export.

In [2]:
# Load from the best available output (346/380 matched from previous run).
df_final = pd.read_csv(
    "cf_data/engle_gaia_xmatch.csv",
    dtype={"gaia_dr3_source_id_from_simbad": "string", "source_id": "string"}
)
print(f"Total rows:          {len(df_final)}")
print(f"With Gaia source_id: {df_final['source_id'].notna().sum()}")
print(f"Missing Gaia data:   {df_final['source_id'].isna().sum()}")

Total rows:          380
With Gaia source_id: 346
Missing Gaia data:   34


In [3]:
# Split missing stars into two groups:
#   1. Known Gaia ID (from SIMBAD) but direct query failed → retry by ID
#   2. No Gaia ID → cone search by SIMBAD coordinates
missing_mask = df_final["source_id"].isna()

retry_direct = df_final[
    missing_mask &
    df_final["gaia_dr3_source_id_from_simbad"].notna()
].drop_duplicates(subset=["gaia_dr3_source_id_from_simbad"])

needs_cone = df_final[
    missing_mask &
    df_final["gaia_dr3_source_id_from_simbad"].isna() &
    df_final["simbad_ra_deg"].notna()
].drop_duplicates(subset=["simbad_ra_deg", "simbad_dec_deg"])

print(f"Have Gaia ID, direct query failed → retry: {len(retry_direct)}")
print(f"No Gaia ID, have SIMBAD coords → cone search: {len(needs_cone)}")
print(f"No SIMBAD match at all (unrecoverable): {df_final[missing_mask & df_final['simbad_ra_deg'].isna()]['star_name'].nunique()}")

Have Gaia ID, direct query failed → retry: 26
No Gaia ID, have SIMBAD coords → cone search: 3
No SIMBAD match at all (unrecoverable): 3


In [4]:
import time

# Step 1: retry direct Gaia query by source_id for the 73 stars
# whose IDs we already know from SIMBAD (previous attempts hit API errors).
retry_core_rows = []
retry_ap_rows   = []
retry_id_failures = []

for i, row in enumerate(retry_direct.itertuples(), start=1):
    sid = str(row.gaia_dr3_source_id_from_simbad)
    print(f"{i}/{len(retry_direct)}: {row.star_name}  (id={sid})")
    try:
        c = query_gaia_one_source_id(sid)
        if len(c) > 0:
            retry_core_rows.append(c)
        a = query_ap_one_source_id(sid)
        if len(a) > 0:
            retry_ap_rows.append(a)
        time.sleep(1.0)
    except Exception as e:
        retry_id_failures.append({"star_name": row.star_name, "source_id": sid, "reason": str(e)})
        print(f"  -> ERROR: {e}")
        time.sleep(2.0)

print(f"\nDirect retry: {len(retry_core_rows)} recovered, {len(retry_id_failures)} still failing")

1/26: GJ 2131 A  (id=4149820284980989184)


2/26: 40 Eri C  (id=3195919254111314816)


3/26: 2MASS J23095781+5506472  (id=1996725077535282944)


4/26: LHS 229  (id=975968718869126144)


5/26: GJ 213  (id=3340477717172813568)


...

23/26: GJ 47  (id=4.266419550437235e+17)


24/26: GJ 9689  (id=1803475188714186368)


25/26: GJ 536  (id=3657653114880309248)


26/26: GJ 4274  (id=2595284016771502080)



Direct retry: 22 recovered, 0 still failing


In [5]:
# Assemble direct retry results and fill into df_final.
if retry_core_rows:
    retry_core = pd.concat(retry_core_rows, ignore_index=True)
    retry_ap   = pd.concat(retry_ap_rows, ignore_index=True) if retry_ap_rows else pd.DataFrame()

    if len(retry_ap) > 0:
        retry_gaia = retry_core.merge(retry_ap, how="left", on="source_id")
    else:
        retry_gaia = retry_core.copy()

    retry_gaia["g_rp"]    = retry_gaia["phot_g_mean_mag"] - retry_gaia["phot_rp_mean_mag"]
    retry_gaia["bp_rp_0"] = retry_gaia["bp_rp"] - retry_gaia["ebpminrp_gspphot"]
    # Keep source_id as plain Python str for reliable lookup
    retry_gaia["source_id_str"] = retry_gaia["source_id"].apply(lambda x: str(int(x)))
    retry_lookup = retry_gaia.set_index("source_id_str")

    gaia_fill_cols = [
        "source_id", "ra", "dec", "parallax", "parallax_error",
        "pmra", "pmdec", "phot_g_mean_mag", "phot_bp_mean_mag",
        "phot_rp_mean_mag", "bp_rp", "g_rp", "ruwe", "ebpminrp_gspphot", "bp_rp_0"
    ]

    # re-extract clean IDs from simbad_ids for matching
    import re as _re
    def _clean_id(ids_str):
        if not isinstance(ids_str, str): return None
        m = _re.search(r"Gaia DR3\s+(\d+)", ids_str)
        return m.group(1) if m else None

    mask = df_final["source_id"].isna() & df_final["simbad_ids"].notna()
    filled = 0
    for idx, row in df_final[mask].iterrows():
        cid = _clean_id(row["simbad_ids"])
        if cid and cid in retry_lookup.index:
            gr = retry_lookup.loc[cid]
            if isinstance(gr, pd.DataFrame): gr = gr.iloc[0]
            for col in gaia_fill_cols:
                if col in gr.index:
                    df_final.at[idx, col] = gr[col]
            df_final.at[idx, "gaia_dr3_source_id_from_simbad"] = cid
            filled += 1
    print(f"Filled {filled} rows from direct retry")
else:
    print("No direct retry results.")

print(f"After direct retry fill:")
print(f"  With Gaia source_id: {df_final['source_id'].notna().sum()}")
print(f"  Still missing:       {df_final['source_id'].isna().sum()}")

TypeError: Invalid value '4.149820284980989e+18' for dtype 'string'. Value should be a string or missing value, got 'float64' instead.

In [ ]:
# Step 2: cone search for stars that have SIMBAD coords but no Gaia ID.
# (These never had a source_id to query directly.)
cone_results  = []
cone_failures = []

for i, row in enumerate(needs_cone.itertuples(), start=1):
    print(f"{i}/{len(needs_cone)}: {row.star_name}")
    try:
        candidates = gaia_cone_search(row.simbad_ra_deg, row.simbad_dec_deg, radius_arcsec=5)
        best, flag = choose_best_gaia_match(candidates)
        if best is not None:
            best["simbad_ra_deg"] = row.simbad_ra_deg
            best["simbad_dec_deg"] = row.simbad_dec_deg
            best["cone_flag"] = flag
            cone_results.append(best)
            print(f"  -> matched source_id={best['source_id']}, dist={best['dist_arcsec']:.2f}'")
        else:
            cone_failures.append({"star_name": row.star_name, "reason": flag})
            print(f"  -> no match ({flag})")
        time.sleep(0.5)
    except Exception as e:
        cone_failures.append({"star_name": row.star_name, "reason": str(e)})
        print(f"  -> ERROR: {e}")
        time.sleep(2.0)

print(f"\nCone search matched: {len(cone_results)}")
print(f"Still unmatched:     {len(cone_failures)}")

In [ ]:
# Inspect cone results before merging
if cone_results:
    cone_df = pd.DataFrame(cone_results)
    cone_df["source_id"] = cone_df["source_id"].astype("string")
    cone_df["g_rp"]    = cone_df["phot_g_mean_mag"] - cone_df["phot_rp_mean_mag"]
    cone_df["bp_rp_0"] = cone_df["bp_rp"] - cone_df["ebpminrp_gspphot"]
    print(cone_df[["source_id", "dist_arcsec", "parallax", "bp_rp", "ruwe", "cone_flag"]].to_string())
else:
    cone_df = pd.DataFrame()
    print("No cone search results.")

In [ ]:
# Fill df_final rows that were missing source_id with cone search results.
# Match on (simbad_ra_deg, simbad_dec_deg) â€” unique per SIMBAD position.
gaia_fill_cols = [
    "source_id", "ra", "dec", "parallax", "parallax_error",
    "pmra", "pmdec", "phot_g_mean_mag", "phot_bp_mean_mag",
    "phot_rp_mean_mag", "bp_rp", "g_rp", "ruwe",
    "ebpminrp_gspphot", "bp_rp_0"
]

if len(cone_df) > 0:
    cone_lookup = cone_df.set_index(["simbad_ra_deg", "simbad_dec_deg"])
    mask = df_final["source_id"].isna() & df_final["simbad_ra_deg"].notna()
    for idx, row in df_final[mask].iterrows():
        key = (row["simbad_ra_deg"], row["simbad_dec_deg"])
        if key in cone_lookup.index:
            cr = cone_lookup.loc[key]
            if isinstance(cr, pd.DataFrame):
                cr = cr.iloc[0]
            for col in gaia_fill_cols:
                if col in cr.index:
                    df_final.at[idx, col] = cr[col]

print(f"After cone search fill:")
print(f"  With Gaia source_id: {df_final['source_id'].notna().sum()}")
print(f"  Still missing:       {df_final['source_id'].isna().sum()}")

In [ ]:
# Manual overrides for names SIMBAD cannot resolve.
MANUAL_NAME_MAP = {
    "Gaia DR3 75525a": "G 148-7",
}
for bad_name, good_name in MANUAL_NAME_MAP.items():
    mask = (df_final["star_name"] == bad_name) & (df_final["simbad_status"] == "no_result")
    if mask.any():
        df_final.loc[mask, "star_name"] = good_name
        print(f"Renamed {bad_name!r} -> {good_name!r} ({mask.sum()} rows)")
import re as _re2, time

# Retry SIMBAD for stars that never matched — strip parentheticals,
# try the GJ/identifier inside brackets first.
def candidate_names(raw):
    names = []
    m = _re2.search(r'\(([^)]+)\)', raw)
    if m: names.append(m.group(1).strip())
    base = _re2.sub(r'\s*\([^)]*\)', '', raw).strip()
    if base and base != raw: names.append(base)
    names.append(raw.strip())
    seen = set()
    return [n for n in names if not (n in seen or seen.add(n))]

no_simbad = df_final[df_final["simbad_status"] == "no_result"].copy()
print(f"Stars with no SIMBAD match to retry: {len(no_simbad)}")

for i, row in enumerate(no_simbad.itertuples(), start=1):
    candidates = candidate_names(row.star_name)
    print(f"\n{i}/{len(no_simbad)}: '{row.star_name}' -> trying {candidates}")
    sim = None
    for cname in candidates:
        try:
            result = simbad_lookup(cname)
            if result and result.get("simbad_status") == "ok":
                sim = result; break
        except: pass
        time.sleep(0.3)
    if sim is None:
        print(f"  -> still no match"); continue
    print(f"  -> {sim['simbad_main_id']}")
    mask = df_final["star_name"] == row.star_name
    df_final.loc[mask, "simbad_main_id"]  = sim["simbad_main_id"]
    df_final.loc[mask, "simbad_ra_deg"]   = sim["simbad_ra_deg"]
    df_final.loc[mask, "simbad_dec_deg"]  = sim["simbad_dec_deg"]
    df_final.loc[mask, "simbad_ids"]      = sim["simbad_ids"]
    df_final.loc[mask, "simbad_status"]   = "ok"
    gaia_id = extract_gaia_dr3_id(sim["simbad_ids"])
    if not gaia_id:
        print(f"  -> no Gaia ID in SIMBAD"); continue
    df_final.loc[mask, "gaia_dr3_source_id_from_simbad"] = gaia_id
    print(f"  -> Gaia ID {gaia_id}, querying...")
    try:
        c = query_gaia_one_source_id(gaia_id)
        a = query_ap_one_source_id(gaia_id)
        if len(c) > 0:
            gr = c.iloc[0]; ap_val = a.iloc[0]["ebpminrp_gspphot"] if len(a) > 0 else None
            for idx in df_final[mask].index:
                df_final.at[idx, "source_id"]        = str(int(gr["source_id"]))
                df_final.at[idx, "ra"]               = gr["ra"]
                df_final.at[idx, "dec"]              = gr["dec"]
                df_final.at[idx, "parallax"]         = gr["parallax"]
                df_final.at[idx, "parallax_error"]   = gr["parallax_error"]
                df_final.at[idx, "pmra"]             = gr["pmra"]
                df_final.at[idx, "pmdec"]            = gr["pmdec"]
                df_final.at[idx, "phot_g_mean_mag"]  = gr["phot_g_mean_mag"]
                df_final.at[idx, "phot_bp_mean_mag"] = gr["phot_bp_mean_mag"]
                df_final.at[idx, "phot_rp_mean_mag"] = gr["phot_rp_mean_mag"]
                df_final.at[idx, "bp_rp"]            = gr["bp_rp"]
                df_final.at[idx, "g_rp"]             = gr["phot_g_mean_mag"] - gr["phot_rp_mean_mag"]
                df_final.at[idx, "ruwe"]             = gr["ruwe"]
                df_final.at[idx, "ebpminrp_gspphot"] = ap_val
                df_final.at[idx, "bp_rp_0"]          = (gr["bp_rp"] - ap_val) if ap_val is not None else None
            print(f"  -> matched! parallax={gr['parallax']:.1f} mas, G={gr['phot_g_mean_mag']:.2f}")
        time.sleep(1.0)
    except Exception as e:
        print(f"  -> Gaia error: {e}"); time.sleep(2.0)

print(f"\nAfter SIMBAD retry fill:")
print(f"  With Gaia source_id: {df_final['source_id'].notna().sum()}")
print(f"  Still missing:       {df_final['source_id'].isna().sum()}")

In [ ]:
# Add RUWE quality flag (>= 1.2 indicates potential binary/astrometric issue).
# Rows with no Gaia data have NaN RUWE; flag is left None for those.
df_final["ruwe_flag"] = df_final["ruwe"].apply(
    lambda r: (r >= 1.2) if pd.notna(r) else None
)

print(f"RUWE >= 1.2 (flagged): {df_final['ruwe_flag'].eq(True).sum()}")
print(f"RUWE < 1.2 (clean):    {df_final['ruwe_flag'].eq(False).sum()}")
print(f"No RUWE data:          {df_final['ruwe_flag'].isna().sum()}")

In [ ]:
print("=== Final Engle x Gaia DR3 cross-match summary ===")
print(f"Total Engle stars:        {len(df_final)}")
print(f"SIMBAD matched:           {(df_final['simbad_status'] == 'ok').sum()}")
print(f"No SIMBAD match:          {(df_final['simbad_status'] == 'no_result').sum()}")
print(f"With Gaia source_id:      {df_final['source_id'].notna().sum()}")
print(f"With parallax:            {df_final['parallax'].notna().sum()}")
print(f"RUWE flagged (>= 1.2):    {df_final['ruwe_flag'].eq(True).sum()}")
print(f"No Gaia match at all:     {df_final['source_id'].isna().sum()}")

out_path = "cf_data/engle_gaia_xmatch.csv"
df_final.to_csv(out_path, index=False)
print(f"\nSaved to {out_path}")